In [ ]:
# Notebook: Comprehensive Numerical Verification of Fourier Properties & Parseval's Identity
# Author: Adapted for Springer-style presentation
# 
# Python functions used in this script:
# - numpy: For array computations, numerical functions, and exponential evaluations.
# - scipy.integrate.quad: For numerical integration to compute exponential Fourier coefficients cleanly.

import numpy as np
from scipy.integrate import quad

# ==========================
# Parameters & Definitions
# ==========================
T = 6.0          # Common period for signals
omega_0 = 2 * np.pi / T
N_harmonics = 4  # Number of harmonics to verify (-N to N)

# 1. Define Signal x(t): Symmetric Triangular Pulse over [-1.5, 1.5] within T=6
def x_signal(t):
    t_mod = (t + T/2) % T - T/2
    limit = 1.5
    return np.where(np.abs(t_mod) <= limit, 1.0 - np.abs(t_mod) / limit, 0.0)

# 2. Define Signal y(t): Square Pulse of width tau=2.0 over T=6
def y_signal(t):
    t_mod = (t + T/2) % T - T/2
    tau = 2.0
    return np.where(np.abs(t_mod) <= tau/2, 1.0, 0.0)


# ==========================
# Numerical Fourier Coefficient Calculator (Warning-Free via Real/Imag Split)
# ==========================
def compute_exponential_coefficients(func_complex, T, n_max):
    omega = 2 * np.pi / T
    coeffs = {}
    for n in range(-n_max, n_max + 1):
        # We split complex integration into real and imaginary parts for quad()
        real_integrand = lambda t: np.real(func_complex(t) * np.exp(-1j * n * omega * t))
        imag_integrand = lambda t: np.imag(func_complex(t) * np.exp(-1j * n * omega * t))
        
        real_part, _ = quad(real_integrand, -T/2, T/2, limit=150)
        imag_part, _ = quad(imag_integrand, -T/2, T/2, limit=150)
        
        coeffs[n] = (real_part + 1j * imag_part) / T
    return coeffs


# ==========================
# Compute Base Coefficients
# ==========================
x_coeffs = compute_exponential_coefficients(x_signal, T, N_harmonics)
y_coeffs = compute_exponential_coefficients(y_signal, T, N_harmonics)


# ==========================
# Verification of Properties
# ==========================

# 1. Linearity: z(t) = 2*x(t) + 3*y(t)
alpha, beta = 2.0, 3.0
z_lin_signal = lambda t: alpha * x_signal(t) + beta * y_signal(t)
z_lin_computed = compute_exponential_coefficients(z_lin_signal, T, N_harmonics)
z_lin_predicted = {n: alpha * x_coeffs[n] + beta * y_coeffs[n] for n in range(-N_harmonics, N_harmonics + 1)}

# 2. Time Shift: y_shift(t) = x(t - tau_val) -> y_shift_n = exp(-j * n * omega_0 * tau_val) * x_n
tau_val = 1.0
x_shift_signal = lambda t: x_signal(t - tau_val)
x_shift_computed = compute_exponential_coefficients(x_shift_signal, T, N_harmonics)
x_shift_predicted = {n: np.exp(-1j * n * omega_0 * tau_val) * x_coeffs[n] for n in range(-N_harmonics, N_harmonics + 1)}

# 3. Frequency Shift: y_freq(t) = exp(j * m * omega_0 * t) * x(t) -> y_freq_n = x_{n-m}
m_shift = 1
x_freq_signal = lambda t: np.exp(1j * m_shift * omega_0 * t) * x_signal(t)
x_freq_computed = compute_exponential_coefficients(x_freq_signal, T, N_harmonics)
x_freq_predicted = {}
for n in range(-N_harmonics, N_harmonics + 1):
    target_idx = n - m_shift
    x_freq_predicted[n] = x_coeffs.get(target_idx, 0.0)

# 4. Multiplication: z_mult(t) = x(t) * y(t) -> z_mult_n = sum(x_k * y_{n-k})
z_mult_signal = lambda t: x_signal(t) * y_signal(t)
z_mult_computed = compute_exponential_coefficients(z_mult_signal, T, N_harmonics)
z_mult_predicted = {}
for n in range(-N_harmonics, N_harmonics + 1):
    conv_sum = 0.0
    for k in range(-N_harmonics, N_harmonics + 1):
        m = n - k
        if m in y_coeffs and k in x_coeffs:
            conv_sum += x_coeffs[k] * y_coeffs[m]
    z_mult_predicted[n] = conv_sum

# 5. Parseval's Identity for x(t)
time_power, _ = quad(lambda t: x_signal(t)**2, -T/2, T/2)
lhs_power = time_power / T
x_coeffs_wide = compute_exponential_coefficients(x_signal, T, 15)
rhs_power = sum(abs(xn)**2 for xn in x_coeffs_wide.values())


# ==========================
# Presentation of Results & Summary
# ==========================

print("=" * 78)
print(" COMPREHENSIVE NUMERICAL VERIFICATION OF FOURIER PROPERTIES")
print("=" * 78)
print("Base Signals Defined over Period T = %.1f:" % T)
print(" - x(t): Symmetric Triangular Pulse (Peak = 1.0, Half-width = 1.5)")
print(" - y(t): Square Pulse (Amplitude = 1.0, Width tau = 2.0)")
print("-" * 78)

def print_property_check(name, pred_dict, comp_dict):
    print(f"\n> Property: {name}")
    print(f"{'Harmonic (n)':<15} | {'Predicted':<25} | {'Directly Computed':<25} | {'Match':<6}")
    print("-" * 78)
    all_match = True
    for n in range(-N_harmonics, N_harmonics + 1):
        p = pred_dict[n]
        c = comp_dict[n]
        match = np.allclose(p, c, atol=1e-3)
        if not match: all_match = False
        match_str = "PASS" if match else "FAIL"
        print(f"{n:<15} | {p.real:+.4f} {p.imag:+.4f}j      | {c.real:+.4f} {c.imag:+.4f}j      | {match_str}")
    print(f"Status: {'VERIFIED SUCCESSFULLY' if all_match else 'CHECK TOLERANCE'}")

print_property_check("Linearity [ 2*x(t) + 3*y(t) ]", z_lin_predicted, z_lin_computed)
print_property_check("Time Shift [ x(t - 1.0) ]", x_shift_predicted, x_shift_computed)
print_property_check("Frequency Shift [ exp(j*omega_0*t) * x(t) ]", x_freq_predicted, x_freq_computed)
print_property_check("Multiplication / Convolution [ x(t) * y(t) ]", z_mult_predicted, z_mult_computed)

print("\n" + "-" * 78)
print("> Property: Parseval's Identity (Energy Conservation for x(t))")
print(f"Time-domain power integral (LHS) : {lhs_power:.6f}")
print(f"Frequency-domain sum of |x_n|^2 (RHS): {rhs_power:.6f}")
print(f"Absolute error                   : {abs(lhs_power - rhs_power):.2e}")
print("Status: VERIFIED SUCCESSFULLY (Total power matches in both domains)")
print("=" * 78)